# 📊 YOLOv8 股票圖形辨識教材

> 用 AI 模型自動辨識 K 線圖中的圖形(W底、M頭、頭肩頂...)

## 🎯 學習目標

1. 認識 YOLOv8 物件偵測模型如何應用在金融圖表
2. 學會用 yfinance 抓股票資料、用 mplfinance 畫 K 線圖
3. 用 HuggingFace Hub 下載預訓練模型(免訓練,直接使用)
4. 體驗 AI 圖形辨識的能力與限制(透過三組對照實驗)

## 🗺️ 完整流程

```
Step 0: 環境準備 (本機裝 Anaconda / Colab 直接用)
Step 1: 安裝套件 (ultralytics、mplfinance、yfinance)
Step 2: 取得 HuggingFace 授權 (申請 Token)
Step 3: 下載 YOLO 模型 (約 87 MB)
Step 4: 認識 6 種圖形標籤
Step 5: 定義核心函式 (抓資料、畫圖、辨識)
Step 6: 實驗 A — 6 檔台股 (6 個月)
Step 7: 實驗 B — 同股票拉長到 1 年
Step 8: 實驗 C — 美股對照組
Step 9: 自訂股票測試
```

## 📋 模型可辨識的 6 種圖形

| 英文標籤 | 中文 | 意義 |
|---------|------|------|
| Head and shoulders bottom | 頭肩底 | 跌→反彈→更跌→反彈→突破,看漲反轉 |
| Head and shoulders top | 頭肩頂 | 漲→回檔→更高→回檔→跌破,看跌反轉 |
| M_Head | M 頭 | 雙頂結構,看跌反轉 |
| W_Bottom | W 底 | 雙底結構,看漲反轉 |
| Triangle | 三角形 | 高低點收斂的整理型態 |
| StockLine | 趨勢線 | 持續性的上升/下降趨勢 |

> ⚠️ **重要提醒**:本教材僅供學習 AI 工具操作,不構成任何投資建議。
> AI 辨識結果可能有誤差,實際投資決策請以人工判斷為主。

## 🚀 在 Colab 一鍵開啟

如果你不想在本機安裝任何東西,直接點下面的按鈕在瀏覽器執行:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

> 將此 .ipynb 上傳到自己的 Google Drive,在 Drive 中右鍵「以 Google Colab 開啟」即可使用。


## Step 0｜環境準備

> 💡 你有兩種方式執行這份教材,擇一即可。

### 方式 A:Google Colab(推薦零基礎學員)

1. 把這份 `.ipynb` 上傳到你的 Google Drive
2. 在檔案上右鍵 → 「選擇開啟工具」→ 「Google Colaboratory」
3. **零安裝、零設定**,直接從 Step 1 開始執行

### 方式 B:本地 Jupyter(進階學員)

1. 前往 👉 https://www.anaconda.com/download 下載 Windows 版(約 900MB)
2. 執行安裝檔,全部按「Next」
3. 安裝完成後,開始選單搜尋「Jupyter Notebook」開啟
4. 瀏覽器跳出檔案目錄後,找到這份 `.ipynb` 點擊開啟

### 兩種環境的差異

| 項目 | Colab | 本地 Jupyter |
|------|-------|------------|
| 安裝難度 | 零安裝 | 需裝 Anaconda |
| 套件管理 | 已預裝多數套件 | 需手動 pip install |
| 模型下載速度 | 快(Google 機房) | 看你的網速 |
| 資料持久性 | 關閉就消失 | 永久保存 |
| 適合對象 | 想快速體驗 | 長期使用、需保留結果 |


## Step 1｜安裝必要套件

> **這格在做什麼**:安裝這次需要的 5 個套件。
> 在 Colab 執行第一次會花 1-2 分鐘,本地通常更快(已裝過會跳過)。

| 套件 | 用途 |
|------|------|
| ultralytics | YOLO 模型載入與推論 |
| mplfinance | 畫 K 線圖 |
| yfinance | 抓股票資料 |
| huggingface_hub | 下載模型權重 |
| openpyxl | 輸出 Excel 報表 |


In [ ]:
# 自動偵測環境並安裝套件
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 偵測到 Google Colab 環境")
    !pip install -q ultralytics mplfinance yfinance huggingface_hub openpyxl
else:
    print("💻 偵測到本地環境")
    !pip install ultralytics mplfinance yfinance huggingface_hub openpyxl

print("✅ 套件安裝完成!")


## Step 2｜取得 HuggingFace 授權

> **這格在做什麼**:這個 YOLO 模型放在 HuggingFace 上,需要登入帳號並接受授權後才能下載。

### 步驟一:接受模型授權

1. 用瀏覽器打開 👉 https://huggingface.co/foduucom/stockmarket-pattern-detection-yolov8
2. 沒帳號就免費註冊一個
3. 在頁面上找到「Agree and access repository」按鈕,按下接受授權

### 步驟二:申請 Access Token

1. 登入後到 👉 https://huggingface.co/settings/tokens
2. 點「Create new token」
3. **Token type 選 `Read`**(只讀就夠)
4. 隨便取個名字(例如 `colab-token`)
5. 按「Generate token」
6. **把產生的 token 複製起來**(類似 `hf_xxxxxxxxxx...`)

> 💡 Token 只會在這個畫面顯示一次,一定要複製!忘了就重新建一個。


## Step 3｜登入 HuggingFace

> **這格在做什麼**:把剛才複製的 token 貼進來,讓程式可以下載模型。
> 為了安全,我們用 `getpass` 隱藏輸入(畫面不會顯示文字)。


In [ ]:
from huggingface_hub import login
from getpass import getpass

# 用 getpass 安全輸入(輸入時看不到文字,正常現象)
hf_token = getpass("請貼上你的 HuggingFace Token (輸入時不會顯示): ")

login(token=hf_token)
print("✅ HuggingFace 登入成功!")


## Step 4｜下載 YOLOv8 模型

> **這格在做什麼**:從 HuggingFace 下載預訓練好的 YOLO 模型(約 87 MB)。
> 第一次下載會花 1-2 分鐘,之後執行會直接跳過。


In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import os

# 下載模型到當前目錄的 model 資料夾
MODEL_DIR = "./model"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = hf_hub_download(
    repo_id="foduucom/stockmarket-pattern-detection-yolov8",
    filename="model.pt",
    local_dir=MODEL_DIR,
)

size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f"✅ 模型下載完成: {model_path}")
print(f"   檔案大小: {size_mb:.2f} MB")

# 載入模型
model = YOLO(model_path)
print(f"✅ 模型載入成功!")
print(f"   可辨識的圖形類別:")
for idx, name in model.names.items():
    print(f"     {idx}: {name}")


## Step 5｜認識 6 種圖形標籤

模型輸出的英文標籤對照中文如下:

| 英文 | 中文 | 多空意義 |
|------|------|---------|
| Head and shoulders bottom | 頭肩底 | 看漲反轉 |
| Head and shoulders top | 頭肩頂 | 看跌反轉 |
| M_Head | M 頭 | 看跌反轉 |
| W_Bottom | W 底 | 看漲反轉 |
| Triangle | 三角形 | 整理型態 |
| StockLine | 趨勢線 | 持續性走勢 |

我們先把對照表寫成 Python 字典,後續會用到。


In [ ]:
# 中英文對照字典
CLASS_NAMES_ZH = {
    'Head and shoulders bottom': '頭肩底',
    'Head and shoulders top': '頭肩頂',
    'M_Head': 'M 頭',
    'StockLine': '趨勢線',
    'Triangle': '三角形',
    'W_Bottom': 'W 底',
}

print("✅ 圖形對照表已建立")
for en, zh in CLASS_NAMES_ZH.items():
    print(f"   {en:30s} → {zh}")


## Step 6｜定義核心函式

> **這格在做什麼**:定義三個函式,把整個流程包裝起來。
> 1. `fetch_and_plot()` — 抓股票資料 + 畫 K 線圖
> 2. `detect_pattern()` — 用 YOLO 辨識圖形
> 3. `scan_stocks()` — 批次掃描多檔股票,**並把標註圖直接顯示在 Notebook 裡**

> 💡 重點:每次跑 `scan_stocks()` 後,你會直接在輸出區看到「YOLO 標註後的 K 線圖」,
> 框框、標籤、信心度都標好了,不用再自己去檔案總管找圖。


In [ ]:
import yfinance as yf
import mplfinance as mpf
import pandas as pd
import os
import glob
from IPython.display import Image, display, Markdown

def fetch_and_plot(ticker, period="6mo", save_dir="./charts"):
    """
    抓股票資料,畫 K 線圖,回傳圖片路徑
    
    參數:
        ticker: 股票代碼,例如 "2330.TW"、"NVDA"
        period: 期間,例如 "6mo"、"1y"
        save_dir: 圖檔儲存資料夾
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # 抓資料
    df = yf.download(ticker, period=period, interval="1d",
                     progress=False, auto_adjust=False)
    
    # 處理 yfinance 新版 multi-level columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # 處理可能的缺值
    df = df.dropna(subset=['Open', 'High', 'Low', 'Close'])
    
    if df.empty or len(df) < 20:
        print(f"  [{ticker}] ⚠️ 資料不足 ({len(df)} 筆),跳過")
        return None
    
    # 檔名安全處理: .TWO 必須在 .TW 之前替換,避免變成 2640O
    safe_name = ticker.replace(".TWO", "_OTC").replace(".TW", "_TW")
    chart_path = os.path.join(save_dir, f"{safe_name}.png")
    
    # 畫 K 線圖
    mpf.plot(df, type='candle', style='charles', volume=True,
             title=f'{ticker} - {period}',
             savefig=dict(fname=chart_path, dpi=100, bbox_inches='tight'),
             figsize=(10, 6))
    
    print(f"  [{ticker}] 取得 {len(df)} 筆 K 線 → 圖檔已存")
    return chart_path


def detect_pattern(model, chart_path, ticker, save_dir="./detect_results", conf=0.25):
    """
    用 YOLO 辨識圖形,回傳偵測結果列表
    
    參數:
        model: YOLO 模型物件
        chart_path: K 線圖路徑
        ticker: 股票代碼(用於命名輸出)
        save_dir: 標註圖儲存資料夾
        conf: 信心度門檻(0~1,越高越嚴格)
    """
    if chart_path is None or not os.path.exists(chart_path):
        return []
    
    safe_name = ticker.replace(".TWO", "_OTC").replace(".TW", "_TW")
    
    results = model.predict(
        source=chart_path, conf=conf, save=True,
        project=save_dir, name=safe_name, exist_ok=True, verbose=False,
    )
    
    detected = []
    if results and len(results) > 0:
        r = results[0]
        if r.boxes is not None and len(r.boxes) > 0:
            for box in r.boxes:
                cls_idx = int(box.cls[0])
                conf_val = float(box.conf[0])
                label_en = model.names[cls_idx]
                label_zh = CLASS_NAMES_ZH.get(label_en, label_en)
                detected.append({
                    'label_en': label_en,
                    'label_zh': label_zh,
                    'confidence': round(conf_val, 3),
                })
    return detected


def scan_stocks(stock_list, period="6mo", conf=0.25, show_images=True):
    """
    批次掃描多檔股票,回傳結果摘要
    
    參數:
        show_images: 是否在 Notebook 中內嵌顯示標註圖(預設 True)
    """
    print(f"\n{'='*60}")
    print(f"批次掃描 {len(stock_list)} 檔股票 (期間: {period}, 門檻: {conf})")
    print(f"{'='*60}\n")
    
    results_summary = []
    label_counter = {}
    
    for ticker in stock_list:
        chart_path = fetch_and_plot(ticker, period=period)
        detected = detect_pattern(model, chart_path, ticker, conf=conf)
        
        if not detected:
            print(f"  [{ticker}] → 未偵測到圖形\n")
            results_summary.append({'ticker': ticker, 'patterns': '(無)'})
            
            # 即使沒偵測到,也顯示原始 K 線圖讓學員自己看
            if show_images and chart_path and os.path.exists(chart_path):
                display(Markdown(f"**[{ticker}] 原始 K 線圖(模型未偵測到圖形)**"))
                display(Image(filename=chart_path, width=600))
        else:
            patterns_str = ", ".join([f"{d['label_zh']}({d['confidence']})" for d in detected])
            print(f"  [{ticker}] → {patterns_str}\n")
            results_summary.append({'ticker': ticker, 'patterns': patterns_str})
            for d in detected:
                label_counter[d['label_zh']] = label_counter.get(d['label_zh'], 0) + 1
            
            # 顯示 YOLO 標註後的圖
            if show_images:
                safe_name = ticker.replace(".TWO", "_OTC").replace(".TW", "_TW")
                # YOLO 預設輸出 .jpg
                annotated_imgs = sorted(
                    glob.glob(f"./detect_results/{safe_name}/*.jpg") +
                    glob.glob(f"./detect_results/{safe_name}/*.png"),
                    key=os.path.getmtime, reverse=True
                )
                if annotated_imgs:
                    display(Markdown(f"**[{ticker}] AI 標註結果:** {patterns_str}"))
                    display(Image(filename=annotated_imgs[0], width=600))
    
    # 統計
    print(f"\n{'='*60}")
    print("各圖形出現次數統計:")
    print(f"{'='*60}")
    if not label_counter:
        print("  (沒有偵測到任何圖形)")
    else:
        for label, count in sorted(label_counter.items(), key=lambda x: -x[1]):
            print(f"  {label}: {count} 次")
    
    return results_summary


print("✅ 核心函式定義完成!")
print("   - fetch_and_plot(): 抓資料 + 畫 K 線")
print("   - detect_pattern(): YOLO 辨識")
print("   - scan_stocks(): 批次掃描 + 內嵌顯示標註圖 🖼️")


## Step 7｜實驗 A:6 檔台股 6 個月

> **這格在做什麼**:用 6 檔台股測試模型,期間設定 6 個月。
> 預期執行時間約 30 秒~1 分鐘(看網路速度)。

### 測試清單

| 代碼 | 公司 |
|------|------|
| 2330.TW | 台積電 |
| 2317.TW | 鴻海 |
| 2640.TWO | 大車隊(注意:.TWO 是櫃買 OTC 股票) |
| 1560.TW | 中砂 |
| 6191.TW | 精成科 |
| 2887.TW | 台新金 |

> 💡 你可以隨時修改清單,例如改成自己的自選股。


In [ ]:
# 實驗 A: 台股 6 個月
exp_a_stocks = [
    "2330.TW",   # 台積電
    "2317.TW",   # 鴻海
    "2640.TWO",  # 大車隊 (.TWO 是櫃買 OTC)
    "1560.TW",   # 中砂
    "6191.TW",   # 精成科
    "2887.TW",   # 台新金
]

result_a = scan_stocks(exp_a_stocks, period="6mo", conf=0.25)


### 🔍 觀察重點

執行完後請打開 `detect_results/` 資料夾,查看每張標註圖。重點看:

1. **框框位置準不準**:模型標出的「W底」、「M頭」位置是否符合圖形定義?
2. **同一張圖會不會重複標**:有時模型會標出 2-3 個重疊的框
3. **信心度與準確度的關係**:0.8 以上一定準嗎?(答案:不一定)

> 💡 建議至少看 3 張標註圖再進入下一步。


## Step 8｜實驗 B:同樣股票拉長到 1 年

> **這格在做什麼**:用同樣的股票清單,但期間從 6 個月改成 1 年。
> 觀察「期間長度」對辨識結果的影響。

這個實驗會驗證一個重要問題:
**同一支股票,只是換期間,辨識結果會一樣嗎?**


In [ ]:
# 實驗 B: 同樣的股票清單,但期間 1 年
result_b = scan_stocks(exp_a_stocks, period="1y", conf=0.25)

# 跟實驗 A 比較
print(f"\n{'='*60}")
print("實驗 A vs 實驗 B 對照(同股票、不同期間)")
print(f"{'='*60}")
print(f"{'股票':12} {'6 個月 (A)':30} {'1 年 (B)':30}")
print("-" * 75)
for a, b in zip(result_a, result_b):
    print(f"{a['ticker']:12} {a['patterns'][:28]:30} {b['patterns'][:28]:30}")


### 🔍 觀察重點

1. **同一支股票結果完全不同?** 這代表模型對「視角範圍」很敏感
2. **K 棒密度改變了**:6 個月約 120 根、1 年約 240 根,K 棒越擠模型越難辨識
3. **實務意義**:使用這類工具時,「期間」是重要參數,不能隨便改

> 💡 從這個實驗可以學到:**AI 工具的結果穩定性是要驗證的**。


## Step 9｜實驗 C:美股對照組

> **這格在做什麼**:用美股測試,因為這個模型是用美股 K 線圖訓練的。
> 觀察「訓練資料分布」對辨識結果的影響。

### 測試清單(對應台股的同產業類別)

| 美股 | 對應台股 | 產業類別 |
|------|---------|---------|
| NVDA | 2330 | 半導體龍頭 |
| DELL | 2317 | 電子代工 |
| SPY | 0050 | 大盤 ETF |
| T | 2412 | 電信 |
| ZIM | 2603 | 海運 |
| XOM | 2002 | 能源/原物料 |


In [ ]:
# 實驗 C: 美股對照組
exp_c_stocks = [
    "NVDA",   # AI 龍頭
    "DELL",   # 戴爾電子
    "SPY",    # S&P 500 ETF
    "T",      # AT&T 電信
    "ZIM",    # 以色列航運
    "XOM",    # Exxon Mobil 埃克森美孚
]

result_c = scan_stocks(exp_c_stocks, period="1y", conf=0.25)


### 🔍 觀察重點

1. **辨識結果有沒有比台股「乾淨」?** (乾淨 = 不會同一張圖標一堆矛盾的框)
2. **6 種圖形都會出現嗎?** 特別注意 Triangle 和 StockLine
3. **跟訓練資料來源相同的市場,辨識力是不是真的比較好?**

> 💡 這個實驗讓你體驗到:**AI 模型的「適用範圍」是有邊界的**,
> 拿訓練資料以外的場景使用,結果可能不如預期。


## Step 10｜自訂股票測試

> **這格在做什麼**:輪到你了!輸入自己的股票清單測試。

### 修改方式

1. 改 `my_stocks` 清單裡的股票代碼
2. 調整 `my_period`(可選 `"3mo"`、`"6mo"`、`"1y"`、`"2y"`)
3. 調整 `my_conf`(0.25 偏寬鬆 / 0.5 適中 / 0.7 嚴格)
4. 執行!

### 股票代碼規則

| 市場 | 代碼格式 | 範例 |
|------|---------|------|
| 美股 | 純英文代碼 | `AAPL`、`TSLA` |
| 台股上市 | 數字 + `.TW` | `2330.TW`、`0050.TW` |
| 台股櫃買 | 數字 + `.TWO` | `2640.TWO` |
| 港股 | 4位數 + `.HK` | `0700.HK`(騰訊) |
| 日股 | 4位數 + `.T` | `7203.T`(豐田) |


In [ ]:
# ✏️ 自由修改下面三個變數

my_stocks = [
    "AAPL",      # Apple
    "TSLA",      # Tesla
    "2454.TW",   # 聯發科
    # 加入你想測試的股票...
]

my_period = "6mo"   # 可改: "3mo"、"6mo"、"1y"、"2y"
my_conf = 0.4       # 可改: 0.25 寬鬆 / 0.4 適中 / 0.6 嚴格

# 執行
result_my = scan_stocks(my_stocks, period=my_period, conf=my_conf)


## 🎉 教材完成!

### 📁 你產生的檔案

執行完所有 Cell 後,以下資料夾會有檔案:

| 資料夾 | 內容 |
|--------|------|
| `./model/` | YOLO 模型權重(`model.pt`) |
| `./charts/` | 各股票的 K 線圖(原圖) |
| `./detect_results/` | YOLO 標註後的圖(有框框那張) |

### 🔄 下次使用

下次再開這份 Notebook,**Step 1~5 一定要重跑**(套件、登入、下載模型、定義函式)。
但模型已經下載過,Step 4 會自動跳過下載,只重新載入。

### 💡 延伸玩法

1. **掃描自選股**:把你的觀察名單放進 Step 10 跑一遍
2. **比較不同期間**:同一支股票跑 3mo/6mo/1y,看哪個結果穩定
3. **調整信心度門檻**:0.7 以上才報告,看看會剩幾個訊號
4. **配合人工判讀**:每次 AI 辨識後,自己也看一次圖,訓練眼力

### 📚 延伸閱讀

- YOLOv8 官方文件:https://docs.ultralytics.com/
- 模型 Model Card:https://huggingface.co/foduucom/stockmarket-pattern-detection-yolov8
- mplfinance 教學:https://github.com/matplotlib/mplfinance

---

> 📌 **重要警語**
> 
> 本教材僅為 AI 工具操作教學,不構成任何投資建議。
> AI 圖形辨識結果有誤差,實際投資決策請以人工判斷為主,並自負盈虧。
